# 12 · Model understanding: which features drive the predictions?
Adapted from the tutorial notebook *Model_understanding* (iris) and the feature-importance part of
*Preprocessing_and_CRISP_DM*. Two DSC-specific steps:
- **Grouping:** one-hot encoding splits e.g. `ZIP_COD` into hundreds of columns. Importances are summed back to the
  **original variable**, otherwise high-cardinality variables look unimportant column by column.
- **Permutation importance** for HistGradientBoosting (it has no built-in importances): shuffle one variable on
  validation data and measure how much macro-F1 drops. This works for any model.

Run after the model notebooks 03, 04, 05 and 08.

In [ ]:
import sys
from pathlib import Path

# find the project root (the folder that contains src/dsc_utils.py) and import the shared helpers
root = Path.cwd()
while not (root / "src" / "dsc_utils.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dsc_utils import *

d = load_prepared()
x_train, x_val, x_test = d["x_train"], d["x_val"], d["x_test"]
y_train, y_val, y_test = d["y_train"], d["y_val"], d["y_test"]
feature_names = d["feature_names"]
print("train", x_train.shape, "| val", x_val.shape, "| test", x_test.shape)
from sklearn.tree import plot_tree
from sklearn.inspection import permutation_importance

original_columns = d["data"].columns
available = {n: load_model(n) for n in ["DT", "LR", "RF", "HGB"] if (OUT / f"model_{n}.pkl").exists()}
print("Loaded:", list(available))

def plot_top(series, title, n=15, xlabel="importance"):
    series.head(n).iloc[::-1].plot(kind="barh", figsize=(8, 0.35 * n + 1))
    plt.title(title); plt.xlabel(xlabel); plt.tight_layout(); plt.show()

## Decision tree
Importance = how much a feature reduces impurity (entropy) over all the splits that use it.

In [ ]:
if "DT" in available:
    DT = available["DT"]
    dt_imp = pd.Series(DT.feature_importances_, index=feature_names).sort_values(ascending=False)
    plot_top(dt_imp, "Decision tree: top encoded features")
    dt_group = group_by_original(dt_imp, original_columns)
    plot_top(dt_group, "Decision tree: importance per original variable")

    plt.figure(figsize=(22, 9))
    plot_tree(DT, max_depth=2, feature_names=feature_names, class_names=list(DT.classes_), filled=True, fontsize=9)
    plt.title("Decision tree: top 2 levels"); plt.show()

## Logistic regression
With 8 classes there is one weight per feature **per class**. All numeric features are standardised, so
weights are comparable. The overall importance below = average absolute weight over the 8 classes.

In [ ]:
if "LR" in available:
    LR = available["LR"]
    coef = pd.DataFrame(LR.coef_, index=LR.classes_, columns=feature_names).reindex(ORDER)
    lr_imp = coef.abs().mean().sort_values(ascending=False)
    plot_top(lr_imp, "Logistic regression: mean |weight| over the 8 classes", xlabel="mean |coefficient|")
    plot_top(group_by_original(lr_imp, original_columns), "Logistic regression: per original variable (summed)",
             xlabel="summed mean |coefficient|")

    # direction: which features push towards A+ and towards G?
    for cls in ["A+", "G"]:
        top = coef.loc[cls].sort_values()
        print(f"\n{cls}: strongest negative weights\n{top.head(5).round(3)}")
        print(f"{cls}: strongest positive weights\n{top.tail(5).round(3)}")

## Random forest
Mean impurity decrease over all trees, with the spread between trees as error bars (as in the tutorial).

In [ ]:
if "RF" in available:
    RF = available["RF"]
    rf_imp = pd.Series(RF.feature_importances_, index=feature_names)
    rf_std = pd.Series(np.std([t.feature_importances_ for t in RF.estimators_], axis=0), index=feature_names)
    top = rf_imp.sort_values(ascending=False).head(15).index[::-1]
    rf_imp[top].plot(kind="barh", xerr=rf_std[top], figsize=(8, 6))
    plt.title("Random forest: top 15 features (error bar = spread between trees)"); plt.xlabel("mean decrease in impurity")
    plt.tight_layout(); plt.show()
    plot_top(group_by_original(rf_imp.sort_values(ascending=False), original_columns),
             "Random forest: importance per original variable")

## HistGradientBoosting: permutation importance
Shuffles one original variable at a time on 5,000 validation buildings and records the drop in macro-F1.
A drop of 0 means the model does not use that variable. This takes a few minutes.

In [ ]:
if "HGB" in available:
    HGB = available["HGB"]
    X_va = d["data"].iloc[d["indices_val"]]
    X_s, y_s = stratified_sample(X_va, y_val, 5000)
    perm = permutation_importance(HGB, X_s, y_s, scoring="f1_macro", n_repeats=3, random_state=0, n_jobs=1)
    hgb_imp = pd.Series(perm.importances_mean, index=X_s.columns).sort_values(ascending=False)
    plot_top(hgb_imp, "HistGradientBoosting: permutation importance", xlabel="drop in validation macro-F1")

## Partial dependence: how does the predicted label change with build year?
For the best model, set every validation building's build year to the same value, predict, and average the
probabilities. Repeated for a range of years, this shows the model's learned effect of age, all else equal.

In [ ]:
if "HGB" in available:
    X_s2 = X_s.copy()
    years = [1915, 1940, 1960, 1980, 2000, 2015, 2023]
    pdp = {}
    for yr in years:
        X_s2["builtyear"] = yr
        pdp[yr] = class_scores(HGB, X_s2).mean().reindex(ORDER)
    pdp = pd.DataFrame(pdp).T
    pdp.plot(kind="area", stacked=True, figsize=(9, 5), colormap="RdYlGn_r")
    plt.title("Average predicted label mix if every building had this build year (HistGradientBoosting)")
    plt.xlabel("build year (set for all buildings)"); plt.ylabel("average predicted probability")
    plt.legend(title="label", bbox_to_anchor=(1.02, 1), loc="upper left"); plt.tight_layout(); plt.show()
    pdp.round(3)

## Do the models agree? Top 10 original variables per model

In [ ]:
tops = {}
if "DT" in available: tops["DT"] = group_by_original(dt_imp, original_columns)
if "LR" in available: tops["LR"] = group_by_original(lr_imp, original_columns)
if "RF" in available: tops["RF"] = group_by_original(rf_imp, original_columns)
if "HGB" in available: tops["HGB"] = hgb_imp
pd.DataFrame({m: s.head(10).index.tolist() for m, s in tops.items()}, index=range(1, 11))

**For the report:** variables that rank high for every model (expected: build year, building type, region,
solar panels, heat loss surface / form factor) are robust drivers of the EPC label. A variable important for only
one model may reflect how that model works rather than a real effect. Importance shows **association**, not causation.

## SHAP values (advanced, extra point in the course)
SHAP splits **one prediction** into contributions per feature: how much each feature pushed this building towards
a label, compared with the average prediction. Averaged over many buildings, it gives a global importance that also
shows the **direction** of each effect. Uses the random forest on 200 validation buildings
(TreeExplainer; install once with `pip install shap`). Takes a few minutes for a large forest.

In [ ]:
import shap

if "RF" in available:
    rf = available["RF"]
    xs_val, ys_val = stratified_sample(x_val, y_val, 200)
    X_dense = xs_val.toarray() if hasattr(xs_val, "toarray") else np.asarray(xs_val)
    explainer = shap.TreeExplainer(rf)
    sv = np.asarray(explainer.shap_values(X_dense, check_additivity=False))
    if sv.ndim == 3 and sv.shape[0] == len(rf.classes_):          # older shap: (classes, rows, features)
        sv = np.transpose(sv, (1, 2, 0))
    print("SHAP array (buildings, features, classes):", sv.shape)

In [ ]:
if "RF" in available:
    # global: mean |SHAP| over buildings and classes, summed per original variable
    shap_imp = pd.Series(np.abs(sv).mean(axis=(0, 2)), index=feature_names)
    plot_top(group_by_original(shap_imp, original_columns), "SHAP: mean |contribution| per original variable",
             xlabel="mean |SHAP value|")

    # direction for one label: what pushes a building towards G?
    g = list(rf.classes_).index("G")
    shap.summary_plot(sv[:, :, g], X_dense, feature_names=list(feature_names), max_display=12, show=False)
    plt.title("SHAP for label G: red = high feature value, right = pushes towards G")
    plt.tight_layout(); plt.show()

In [ ]:
if "RF" in available:
    # local: why did the model predict this label for one building?
    i = 0
    cls = rf.predict(X_dense[i:i + 1])[0]
    c = list(rf.classes_).index(cls)
    contrib = pd.Series(sv[i, :, c], index=feature_names)
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(10)
    print(f"Building {i}: true label {ys_val[i]}, predicted {cls}")
    top.iloc[::-1].plot(kind="barh", color=np.where(top.iloc[::-1] > 0, "tab:red", "tab:blue"), figsize=(8, 4))
    plt.title(f"Top 10 feature contributions towards {cls} (red = towards, blue = away)"); plt.xlabel("SHAP value")
    plt.tight_layout(); plt.show()